In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

In [2]:
# loade text file
loader = TextLoader("langchain_sample.txt")
row_docs = loader.load()

# split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size =500, chunk_overlap = 50)
docs = splitter.split_documents(row_docs)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [3]:
# user query
query = "How can i use langchain to build an application with memory and tools?"

In [7]:
# FAISS and Hugging face model for embeddings

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs,embedding_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 438.22it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
retriver = vectorstore.as_retriever(search_kwargs={"k":8})

In [10]:
# prompt and use the llm
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
llm=init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq"
)
llm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000014F9E7D9970>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000014F9DEFB020>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""") 

In [12]:
retrive_docs = retriver.invoke(query)

In [13]:
retrive_docs

[Document(id='e4077c0b-e834-43eb-8587-54cab4790dc4', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='25693597-847b-4549-a90d-e12be99ae236', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='7e11d6bc-0e71-4c99-a1be-4711c492aeb1', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [14]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000014F9E7D9970>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000014F9DEFB020>, model_name='ll

In [16]:
doc_lines = [f"{i+1}. {doc.page_content}" for i,doc in enumerate(retrive_docs)]
formatted_docs = "\n".join(doc_lines)

In [20]:
doc_lines

['1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and comp

In [19]:
formatted_docs

'1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed ind

In [21]:
response  = chain.invoke({"question":query,"documents":formatted_docs})

In [22]:
response

"To rank the documents from most to least relevant, I will analyze each document in relation to the user's question about using LangChain to build an application with memory and tools.\n\nAfter examining the documents, I would rank them as follows:\n\n0,1,2,5,4,3\n\nHere's my reasoning:\n\n1. Document 1 directly addresses the user's question, mentioning LangChain's support for memory and tool integration. It's highly relevant to the user's query.\n\n2. Document 2 provides a general overview of LangChain, mentioning its components, including memory and tools. Although it doesn't directly address the user's question, it provides valuable context.\n\n3. Document 5 discusses Retrieval-Augmented Generation (RAG) and how LangChain makes it easy to implement. Although RAG is not directly related to the user's question, it shows LangChain's capabilities in integrating various techniques, making it somewhat relevant.\n\n4. Document 4 is about FAISS, a library unrelated to LangChain's primary fu

In [24]:
# step 5 : parse and rerank
indices = [int(x.strip()) -1 for x in response.split(",") if x.strip().isdigit()]
reranked_docs = [retrive_docs[i] for i in indices if 0<=i<len(retrive_docs)]

In [26]:
# show results
print("Final Reranked Results: ")
for i,doc in enumerate(reranked_docs,1):
    print(f"\nRank {i}: \n {doc.page_content}")

Final Reranked Results: 

Rank 1: 
 LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 2: 
 LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 3: 
 Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.
BM25 is a traditional sparse retrieval method that scores documents based on keyword matching. Although fast, it 